# HONOR 200 Pro — ADB Backup
**Source:** `/sdcard/` (phone internal storage)  
**Destination:** `E:\HonorPhoneBackup22MAY2026`

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import subprocess, os, time
from datetime import datetime
from IPython.display import clear_output

DEST   = r"E:\HonorPhoneBackup22MAY2026"
SOURCE = "/sdcard/"

os.makedirs(DEST, exist_ok=True)

# Confirm ADB sees the phone
result = subprocess.run(["adb", "devices"], capture_output=True, text=True)
print(result.stdout)

if "device" not in result.stdout.split("List of devices")[1]:
    raise RuntimeError("No authorised device found. Check USB debugging is on and tap Allow on the phone.")

print(f"Destination : {DEST}")
print(f"Started     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# ── Cell 2: List top-level folders on phone ───────────────────────────────────
result = subprocess.run(["adb", "shell", "ls", "/sdcard/"], capture_output=True, text=True)
print("Phone internal storage — top level:")
print(result.stdout)

In [ ]:
# ── Cell 3: Run backup with live progress ─────────────────────────────────────
# adb pull streams one line per file — we capture and display a running summary.

print(f"Pulling {SOURCE} → {DEST}")
print("This will take a while. Watch the counter below.\n")

start      = time.time()
file_count = 0
skip_count = 0
err_count  = 0
last_file  = ""

proc = subprocess.Popen(
    ["adb", "pull", SOURCE, DEST],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace"
)

for line in proc.stdout:
    line = line.strip()
    if not line:
        continue
    if line.startswith("pull:"):
        file_count += 1
        last_file = line[6:60]   # trim for display
    elif "skipped" in line.lower():
        skip_count += 1
    elif line.startswith("error") or "failed" in line.lower():
        err_count += 1

    # Refresh display every 25 files
    if file_count % 25 == 0:
        elapsed = time.time() - start
        clear_output(wait=True)
        print(f"Files pulled : {file_count:,}")
        print(f"Skipped      : {skip_count:,}")
        print(f"Errors       : {err_count:,}")
        print(f"Elapsed      : {elapsed/60:.1f} min")
        print(f"Last file    : {last_file}")

proc.wait()
elapsed = time.time() - start
clear_output(wait=True)
print("=" * 45)
print("  BACKUP COMPLETE" if proc.returncode == 0 else f"  FINISHED (exit code {proc.returncode})")
print("=" * 45)
print(f"  Files pulled : {file_count:,}")
print(f"  Skipped      : {skip_count:,}")
print(f"  Errors       : {err_count:,}")
print(f"  Duration     : {elapsed/60:.1f} min")
print(f"  Finished     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 45)

In [ ]:
# ── Cell 4: Verify backup on disk ─────────────────────────────────────────────
count = 0
size  = 0
for root, _, files in os.walk(DEST):
    for f in files:
        try:
            size += os.path.getsize(os.path.join(root, f))
            count += 1
        except OSError:
            pass

top_dirs = [d for d in os.listdir(DEST) if os.path.isdir(os.path.join(DEST, d))]

print("=" * 45)
print("  DISK VERIFICATION")
print("=" * 45)
print(f"  Location : {DEST}")
print(f"  Folders  : {len(top_dirs)}")
print(f"  Files    : {count:,}")
print(f"  Size     : {size/1024**3:.2f} GB")
print("=" * 45)

# TikTok — Cookie Export + Auto-Unfollow

In [ ]:
# ── TikTok Step 1: Clipboard watcher — click Export in Cookie-Editor then come back ──
import subprocess, json, time
from pathlib import Path
from IPython.display import clear_output

OUT = Path(r"C:\Users\equat\Downloads\tiktok_cookies.json")

print("WAITING FOR COOKIES...")
print("→ Go to tiktok.com in Chrome")
print("→ Click Cookie-Editor icon")
print("→ Click Export")
print("→ Come back here — file saves automatically\n", flush=True)

last = ""
while True:
    result = subprocess.run(["powershell", "-command", "Get-Clipboard"],
                            capture_output=True, text=True, timeout=5)
    clip = result.stdout.strip()
    if clip and clip != last and clip.startswith("["):
        try:
            data = json.loads(clip)
            tiktok = [c for c in data if "tiktok" in c.get("domain", "").lower()]
            if len(tiktok) > 0:
                OUT.write_text(clip, encoding="utf-8")
                clear_output(wait=True)
                print(f"✓ SAVED — {len(tiktok)} TikTok cookies → {OUT}")
                break
        except json.JSONDecodeError:
            pass
    last = clip
    time.sleep(1)

In [ ]:
# ── TikTok Step 2: Auto-unfollow everyone ─────────────────────────────────────
import json, requests, time, sys
from datetime import datetime

COOKIES_FILE = r"C:\Users\equat\Downloads\tiktok_cookies.json"
LOG_FILE     = r"E:\SunoMaster\scripts\tiktok_unfollow_log.txt"
DELAY        = 1.5

def log(msg):
    line = f"[{datetime.now().strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")

with open(COOKIES_FILE) as f:
    raw = json.load(f)
cookies = {c["name"]: c["value"] for c in raw}

session = requests.Session()
session.cookies.update(cookies)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": "https://www.tiktok.com/",
    "Origin": "https://www.tiktok.com",
}

def get_following(cursor=0, count=200):
    url = "https://www.tiktok.com/api/relation/following/list/"
    params = {"count": count, "cursor": cursor, "scene": 21,
              "msToken": cookies.get("msToken", "")}
    r = session.get(url, headers=HEADERS, params=params, timeout=15)
    return r.json()

def unfollow_user(user_id, sec_uid):
    url = "https://www.tiktok.com/api/commit/follow/user/"
    params = {"user_id": user_id, "sec_user_id": sec_uid, "type": 2,
              "msToken": cookies.get("msToken", "")}
    r = session.post(url, headers=HEADERS, params=params, timeout=15)
    return r.json()

# Fetch full following list
log("Fetching following list...")
all_following = []
cursor = 0
while True:
    data = get_following(cursor=cursor)
    users = data.get("userList", [])
    if not users:
        break
    for u in users:
        ui = u.get("user", {})
        all_following.append({"user_id": ui.get("id"), "sec_uid": ui.get("secUid"),
                               "nickname": ui.get("nickname", "?"), "unique_id": ui.get("uniqueId", "?")})
    has_more = data.get("hasMore", False)
    cursor = data.get("cursor", 0)
    log(f"  Fetched {len(all_following)} so far...")
    if not has_more:
        break
    time.sleep(0.5)

total = len(all_following)
log(f"Total following: {total}")

done, failed = 0, []
for i, user in enumerate(all_following, 1):
    try:
        result = unfollow_user(user["user_id"], user["sec_uid"])
        if result.get("status_code") == 0:
            done += 1
            log(f"[{i}/{total}] UNFOLLOWED: @{user['unique_id']}")
        else:
            failed.append(user["unique_id"])
            log(f"[{i}/{total}] FAILED (status {result.get('status_code')}): @{user['unique_id']}")
    except Exception as e:
        failed.append(user["unique_id"])
        log(f"[{i}/{total}] ERROR: @{user['unique_id']} — {e}")
    time.sleep(DELAY)

log(f"\n=== DONE — Unfollowed: {done}/{total} | Failed: {len(failed)} ===")